# Raw harvest: AniList + Kitsu + Shikimori
Lands data locally in `../data/raw/` as JSONL (one record per line, each with `_loaded_at`). BigQuery load and dbt come later.

| Step | Source | What we take | Calls |
|---|---|---|---|
| 1 | **AniList** (GraphQL) | full catalog: titles, dates, genres, studios, relations, top characters. Gives `idMal`. | ~400 pages |
| 2 | **Kitsu** (JSON:API) | `id`, `subtype`, `titles.en`, `canonicalTitle`, `ageRating`, `ageRatingGuide` | 1 per 20 MAL ids |
| 3 | **Shikimori** (GraphQL) | `id`, `malId`, `name`, `english`, `score`, `franchise`, `studios` | 1 per 50 MAL ids |
| 4 | **Shikimori** (REST) | franchise graph: one call per distinct `franchise` slug | 1 per franchise |

Kitsu and Shikimori are looked up by **MAL id** (from AniList `idMal`), so AniList runs first. Jikan was dropped: its public API is down.

**Rate limits** (one shared limiter per host, see below)
- AniList: 30 req/min (the headers say how many are left).
- Shikimori: 5 req/s **and** 90 req/min, shared by REST and GraphQL. We stay at ~85/min.
- Kitsu: no published limit, it throttles heavy clients. We stay polite at ~2.5 req/s and back off on 429.

Every step is **resumable**: re-run a cell and it skips what is already landed.

In [5]:
import json, time, pathlib, datetime as dt
import requests
import pandas as pd

RAW = pathlib.Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

ANILIST_OUT = RAW / "anilist_media.jsonl"
ANILIST_STATE = RAW / "anilist_state.json"
KITSU_OUT = RAW / "kitsu_anime.jsonl"
KITSU_MISSING = RAW / "kitsu_missing.jsonl"          # mal_ids Kitsu has no mapping for, so we don't retry them
SHIKI_OUT = RAW / "shikimori_anime.jsonl"
SHIKI_MISSING = RAW / "shikimori_missing.jsonl"
SHIKI_FRANCHISE_OUT = RAW / "shikimori_franchise.jsonl"

def now():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def append_jsonl(path, records):
    with path.open("a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl(path):
    if not path.exists():
        return
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


class RateLimiter:
    """Blocks so that calls are at least `min_interval` seconds apart."""
    def __init__(self, min_interval):
        self.min_interval = min_interval
        self._last = 0.0

    def wait(self):
        delta = time.monotonic() - self._last
        if delta < self.min_interval:
            time.sleep(self.min_interval - delta)
        self._last = time.monotonic()


def request_with_backoff(method, url, limiter, retries=6, **kwargs):
    """Rate-limited request. Retries 429 / 5xx / network errors with backoff.
    Returns the response for anything else (including 4xx, so the caller decides what a 404 means)."""
    for attempt in range(retries):
        limiter.wait()
        try:
            r = requests.request(method, url, timeout=60, **kwargs)
        except requests.RequestException as e:
            print(f"{type(e).__name__} on {url}, retrying")
            time.sleep(10 * (attempt + 1))
            continue
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 30)) + 1
            print(f"429 from {url.split('/')[2]}, sleeping {wait}s")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(10 * (attempt + 1))
            continue
        return r
    raise RuntimeError(f"{url} failed after {retries} attempts")


def batches(items, n):
    for i in range(0, len(items), n):
        yield items[i:i + n]


## 1. AniList

In [ ]:
ANILIST_URL = "https://graphql.anilist.co"
PER_PAGE = 50
MAX_PAGES_PER_PARTITION = 100 
ANILIST_QUERY = """
query ($page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    media(type: ANIME, __FILTERS__ sort: __SORT__) {
      id idMal
      title { english }
      format status season seasonYear
      startDate { year month day } endDate { year month day }
      episodes duration source countryOfOrigin isAdult
      genres
      averageScore popularity favourites
      studios { edges { isMain node { id name isAnimationStudio } } }
      relations { edges { relationType node { id idMal type format title { english } } } }
      characters(sort: FAVOURITES_DESC, perPage: 25) {
        edges { role node { id name { full native } favourites } }
      }
    }
  }
}
"""

def anilist_page(query, page, retries=6):
    for attempt in range(retries):
        r = requests.post(ANILIST_URL, json={"query": query, "variables": {"page": page, "perPage": PER_PAGE}}, timeout=60)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 60)) + 1
            print(f"429, sleeping {wait}s")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(5 * (attempt + 1))
            continue
        r.raise_for_status()
        remaining = int(r.headers.get("X-RateLimit-Remaining", 30))
        return r.json()["data"]["Page"]["media"], remaining
    raise RuntimeError(f"AniList page {page} failed after {retries} attempts")

this_year = dt.date.today().year
PARTITIONS = (
    [(f"year={y}", f"seasonYear: {y},", "ID") for y in range(1910, this_year + 3)]
    + [("not_yet_released", "status: NOT_YET_RELEASED,", "ID"),
       ("newest_ids", "", "ID_DESC")]
)

state = json.loads(ANILIST_STATE.read_text()) if ANILIST_STATE.exists() else {}
done_parts = set(state.get("done", []))
seen_ids = {m["id"] for m in read_jsonl(ANILIST_OUT)}   # skip entries already landed (also from the old by-page run)
print(len(seen_ids), "entries already landed")

for name, filters, sort in PARTITIONS:
    if name in done_parts:
        continue
    query = ANILIST_QUERY.replace("__FILTERS__", filters).replace("__SORT__", sort)
    new = 0
    for page in range(1, MAX_PAGES_PER_PARTITION + 1):
        media, remaining = anilist_page(query, page)
        if not media:
            break
        fresh = [m for m in media if m["id"] not in seen_ids]
        append_jsonl(ANILIST_OUT, ({**m, "_loaded_at": now()} for m in fresh))
        seen_ids.update(m["id"] for m in fresh)
        new += len(fresh)
        time.sleep(61 if remaining <= 2 else 2.1)   # AniList allows ~30 req/min
    done_parts.add(name)
    ANILIST_STATE.write_text(json.dumps({"done": sorted(done_parts)}))   # checkpoint per finished partition
    print(f"{name}: {new} new entries (total {len(seen_ids)})")
print("done")

5000 entries already landed
429, sleeping 61s
year=1910: 361 new entries (total 5361)
year=1911: 449 new entries (total 5810)
year=1912: 479 new entries (total 6289)
year=1913: 480 new entries (total 6769)
year=1914: 553 new entries (total 7322)
year=1915: 513 new entries (total 7835)
year=1916: 580 new entries (total 8415)
year=1917: 549 new entries (total 8964)
year=1918: 522 new entries (total 9486)
year=1919: 462 new entries (total 9948)
year=1920: 431 new entries (total 10379)
year=1921: 506 new entries (total 10885)
year=1922: 474 new entries (total 11359)
year=1923: 496 new entries (total 11855)
year=1924: 468 new entries (total 12323)
year=1925: 482 new entries (total 12805)
year=1926: 452 new entries (total 13257)
year=1927: 80 new entries (total 13337)
year=1928: 1 new entries (total 13338)
year=1929: 0 new entries (total 13338)
year=1930: 0 new entries (total 13338)
year=1931: 0 new entries (total 13338)
year=1932: 0 new entries (total 13338)
year=1933: 0 new entries (total 

In [ ]:
al = pd.DataFrame(read_jsonl(ANILIST_OUT))
print(len(al), "rows,", al["id"].nunique(), "unique AniList ids")
print("with idMal:", al["idMal"].notna().sum())
al[["id", "idMal", "format", "status", "seasonYear", "popularity"]].head()

## 2. Target list
MAL ids to look up in Kitsu and Shikimori: AniList entries with an `idMal`, `MUSIC` excluded, one row per MAL id, most popular first (so stopping early still leaves the useful titles). The raw AniList file is left untouched.

In [6]:
MAL_LIMIT = None   # int = only the N most popular titles (good for a test run); None = all

al_all = pd.DataFrame(read_jsonl(ANILIST_OUT))
print(al_all["format"].value_counts(dropna=False).to_dict())
mal_ids = (
    al_all[al_all["format"] != "MUSIC"][["idMal", "popularity"]]
    .dropna(subset=["idMal"])
    .drop_duplicates("idMal")
    .sort_values("popularity", ascending=False)["idMal"].astype(int).tolist()
)
if MAL_LIMIT:
    mal_ids = mal_ids[:MAL_LIMIT]
print(len(mal_ids), "MAL ids to look up")

{'TV': 4886, 'OVA': 3761, 'MOVIE': 2791, 'ONA': 2319, 'SPECIAL': 1615, 'TV_SHORT': 1221, 'MUSIC': 1199, nan: 85}
15845 MAL ids to look up


## 3. Kitsu
Kitsu keeps a `/mappings` table, so one call looks up **20 MAL ids at once** and (with `include=item`) returns the anime itself. Sparse fieldsets (`fields[anime]`) keep only what we want:
`id`, `subtype` (TV / movie / ...), `titles.en`, `canonicalTitle`, `ageRating`, `ageRatingGuide`. We also store `mal_id` as the join key.
Not every MAL id has a Kitsu entry; those go to `kitsu_missing.jsonl`.

In [7]:
KITSU_URL = "https://kitsu.io/api/edge/mappings"
KITSU_BATCH = 20                       # Kitsu's max page size
kitsu_limiter = RateLimiter(0.4)       # ~2.5 req/s, no official limit
KITSU_HEADERS = {"Accept": "application/vnd.api+json", "User-Agent": "animdle/0.1"}

def kitsu_lookup(batch):
    """MAL ids -> {mal_id: trimmed Kitsu record}"""
    found, offset = {}, 0
    while True:
        r = request_with_backoff("GET", KITSU_URL, kitsu_limiter, headers=KITSU_HEADERS, params={
            "filter[externalSite]": "myanimelist/anime",
            "filter[externalId]": ",".join(map(str, batch)),
            "include": "item",
            "fields[mappings]": "externalId,item",
            "fields[anime]": "titles,canonicalTitle,ageRating,ageRatingGuide,subtype",
            "page[limit]": KITSU_BATCH, "page[offset]": offset,
        })
        r.raise_for_status()
        body = r.json()
        items = {(i["type"], i["id"]): i for i in body.get("included", [])}
        for m in body["data"]:
            ref = m["relationships"]["item"]["data"]
            item = items.get((ref["type"], ref["id"]))
            if ref["type"] != "anime" or item is None:
                continue
            a = item["attributes"]
            found[int(m["attributes"]["externalId"])] = {
                "mal_id": int(m["attributes"]["externalId"]),
                "id": item["id"],
                "subtype": a.get("subtype"),
                "title_en": (a.get("titles") or {}).get("en"),
                "canonical_title": a.get("canonicalTitle"),
                "age_rating": a.get("ageRating"),
                "age_rating_guide": a.get("ageRatingGuide"),
            }
        if "next" not in body.get("links", {}):
            return found
        offset += KITSU_BATCH

done = {r["mal_id"] for r in read_jsonl(KITSU_OUT)} | {r["mal_id"] for r in read_jsonl(KITSU_MISSING)}
todo = [i for i in mal_ids if i not in done]
print(f"{len(todo)} to fetch in {-(-len(todo) // KITSU_BATCH)} requests (~{len(todo) / KITSU_BATCH * 0.4 / 60:.0f} min)")

for n, batch in enumerate(batches(todo, KITSU_BATCH), 1):
    found = kitsu_lookup(batch)
    ts = now()
    append_jsonl(KITSU_OUT, ({**rec, "_loaded_at": ts} for rec in found.values()))
    append_jsonl(KITSU_MISSING, ({"mal_id": i, "_loaded_at": ts} for i in batch if i not in found))
    if n % 50 == 0:
        print(f"{n * KITSU_BATCH}/{len(todo)}")
print("done")

15845 to fetch in 793 requests (~5 min)
1000/15845
2000/15845
3000/15845
4000/15845
5000/15845
6000/15845
7000/15845
8000/15845
9000/15845
10000/15845
11000/15845
12000/15845
13000/15845
14000/15845
15000/15845
done


## 4. Shikimori: anime
Shikimori ids are MAL ids. Its GraphQL endpoint takes `ids: "1,5,6,..."`, so **50 anime per call** instead of one REST call each (REST would be 90/min, about 3 hours for everything; this is a few minutes).
Fields: `id`, `malId`, `name`, `english`, `score`, `franchise` (slug that groups all seasons/movies of a franchise), `studios { id name }`.
GraphQL studios only expose `name` (the REST `filtered_name` variant is not available here). Ids not returned go to `shikimori_missing.jsonl`.

In [8]:
SHIKI_GRAPHQL = "https://shikimori.one/api/graphql"
SHIKI_BATCH = 50
shiki_limiter = RateLimiter(0.7)       # ~85 req/min, under the 90/min cap (5 req/s is never an issue at this pace)
SHIKI_HEADERS = {"User-Agent": "animdle/0.1"}

SHIKI_QUERY = """
query ($ids: String!, $limit: PositiveInt!) {
  animes(ids: $ids, limit: $limit) {
    id malId name english score franchise
    studios { id name }
  }
}
"""

def shiki_lookup(batch):
    r = request_with_backoff("POST", SHIKI_GRAPHQL, shiki_limiter, headers=SHIKI_HEADERS,
                             json={"query": SHIKI_QUERY, "variables": {"ids": ",".join(map(str, batch)), "limit": SHIKI_BATCH}})
    r.raise_for_status()
    body = r.json()
    if body.get("errors"):
        raise RuntimeError(body["errors"])
    return body["data"]["animes"]

done = {r["mal_id"] for r in read_jsonl(SHIKI_OUT)} | {r["mal_id"] for r in read_jsonl(SHIKI_MISSING)}
todo = [i for i in mal_ids if i not in done]
print(f"{len(todo)} to fetch in {-(-len(todo) // SHIKI_BATCH)} requests (~{len(todo) / SHIKI_BATCH * 0.7 / 60:.0f} min)")

for n, batch in enumerate(batches(todo, SHIKI_BATCH), 1):
    animes = shiki_lookup(batch)
    ts = now()
    got = set()
    out = []
    for a in animes:
        mal_id = int(a["malId"] or a["id"])
        got.add(mal_id)
        out.append({"mal_id": mal_id, "id": int(a["id"]), "name": a["name"], "english": a["english"], "score": a["score"],
                    "franchise": a["franchise"], "studios": [{"id": int(s["id"]), "name": s["name"]} for s in a["studios"]],
                    "_loaded_at": ts})
    append_jsonl(SHIKI_OUT, out)
    append_jsonl(SHIKI_MISSING, ({"mal_id": i, "_loaded_at": ts} for i in batch if i not in got))
    if n % 20 == 0:
        print(f"{n * SHIKI_BATCH}/{len(todo)}")
print("done")

15845 to fetch in 317 requests (~4 min)
1000/15845
2000/15845
3000/15845
4000/15845
5000/15845
6000/15845
7000/15845
8000/15845
9000/15845
10000/15845
11000/15845
12000/15845
13000/15845
14000/15845
15000/15845
done


## 5. Shikimori: franchise graph
Each anime has a `franchise` slug (e.g. `cowboy_bebop`). The graph endpoint returns **every member of that franchise and how they connect**, so we call it **once per distinct slug**, not once per anime, using any member's id.

What the endpoint returns, and what we keep:
- `nodes[]`: one per anime in the franchise. **Keep** `id` (= MAL id, joins to everything else), `year`, `date` (unix time of first air date, for ordering). Dropped: Russian `name`/`kind`, image and url (we get names from AniList).
- `links[]`: one per edge. **Keep** `source_id` and `target_id` (anime ids, i.e. MAL ids) and `relation` (`sequel`, `side_story`, `parent_story`, ...). Dropped: `id` (Shikimori's internal row id for the edge, meaningless to us), `source` / `target` (positions in the `nodes` array, only used for drawing the graph) and `weight` (drawing too).
- Each stored row also has the `franchise` slug and `current_id` (the anime we asked with), so every graph roots back to its franchise.

One call per franchise, at ~85/min. Franchises that 404 are stored with their status code.

In [9]:
SHIKI_REST = "https://shikimori.one/api"

# one representative anime per franchise slug (the most popular, since mal_ids is ordered by popularity)
rank = {m: n for n, m in enumerate(mal_ids)}
rep = {}
for rec in read_jsonl(SHIKI_OUT):
    slug = rec["franchise"]
    if slug and (slug not in rep or rank.get(rec["mal_id"], 1e9) < rank.get(rep[slug], 1e9)):
        rep[slug] = rec["mal_id"]
slugs = sorted(rep, key=lambda s: rank.get(rep[s], 1e9))

done = {r["franchise"] for r in read_jsonl(SHIKI_FRANCHISE_OUT)}
todo = [s for s in slugs if s not in done]
print(f"{len(slugs)} franchises, {len(todo)} to fetch (~{len(todo) * 0.7 / 60:.0f} min)")

for n, slug in enumerate(todo, 1):
    r = request_with_backoff("GET", f"{SHIKI_REST}/animes/{rep[slug]}/franchise", shiki_limiter, headers=SHIKI_HEADERS)
    ts = now()
    if r.status_code >= 400:
        append_jsonl(SHIKI_FRANCHISE_OUT, [{"franchise": slug, "current_id": rep[slug], "status_code": r.status_code, "nodes": [], "links": [], "_loaded_at": ts}])
        continue
    g = r.json()
    append_jsonl(SHIKI_FRANCHISE_OUT, [{
        "franchise": slug,
        "current_id": g["current_id"],
        "nodes": [{"id": nd["id"], "year": nd.get("year"), "date": nd.get("date")} for nd in g["nodes"]],
        "links": [{"source_id": l["source_id"], "target_id": l["target_id"], "relation": l["relation"]} for l in g["links"]],
        "_loaded_at": ts,
    }])
    if n % 100 == 0:
        print(f"{n}/{len(todo)}")
print("done")

2751 franchises, 2751 to fetch (~32 min)
100/2751
200/2751
300/2751
400/2751
500/2751
600/2751
700/2751
800/2751
900/2751
1000/2751
1100/2751
1200/2751
1300/2751
1400/2751
1500/2751
1600/2751
1700/2751
1800/2751
1900/2751
ConnectionError on https://shikimori.one/api/animes/913/franchise, retrying
2000/2751
ConnectionError on https://shikimori.one/api/animes/5997/franchise, retrying
2100/2751
2200/2751
ConnectionError on https://shikimori.one/api/animes/4199/franchise, retrying
ConnectionError on https://shikimori.one/api/animes/60089/franchise, retrying
2300/2751
2400/2751
2500/2751
2600/2751
2700/2751
done
